# 04 - Embeddings Notebook (updated 08/02)

Pulling in pre-processed data

### Load Dependencies and Data

In [ ]:
# Set dependencies
import pandas as pd
import numpy as np
import torch
from pyprojroot import here
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm


# Project path anchors
REPO_ROOT = here()
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"

SEED = 1234

### Single-turn: Build from Raw WildGuardMix

In [ ]:
# Load the preprocessed single-turn splits (same files as 06_flo_baseline)
def load_split(split):
    X = pd.read_csv(PROCESSED_DATA_DIR / f"singleturn_X_{split}.csv")
    y = pd.read_csv(PROCESSED_DATA_DIR / f"singleturn_Y_{split}.csv")
    df = X.copy()
    df["harm"] = y.iloc[:, 0]
    df["split"] = split
    return df

singleturn = pd.concat(
    [load_split(s) for s in ("train", "val", "test")], ignore_index=True
)

print(singleturn["split"].value_counts())
singleturn.head(3)

In [ ]:
# Sanity checks on the preprocessed data
assert not singleturn["conversation"].isna().any()
assert not singleturn["conversation"].str.strip().eq("").any()
assert singleturn["harm"].isin([0, 1]).all()

# class balance per split (~50/50)
print(singleturn.groupby("split")["harm"].mean())

In [ ]:
singleturn.to_parquet(PROCESSED_DATA_DIR / "singleturn_all.parquet", index=False)

### Single-Turn: Generate embeddings

#### GPT-2 Embeddings


In [ ]:
# GPT -2 embeddings as baseline

# check to make sure the file doesn't exist
# this setup is for a GPU, CPU's will take extra long

out_path = PROCESSED_DATA_DIR / "singleturn_emb_gpt2.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModel.from_pretrained("gpt2").to(device)
    model.eval()

    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(singleturn), 16)):
            batch = singleturn["conversation"].iloc[i:i + 16].tolist()
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
            outputs = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1)
            pooled = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            all_embeddings.append(pooled.cpu().numpy())

    np.save(out_path, np.concatenate(all_embeddings, axis=0))


#### Qwen 3

In [ ]:
# Qwen 3

# check to make sure the file doesn't exist
# this setup is for a GPU, CPU's will take extra long

out_path = PROCESSED_DATA_DIR / "singleturn_emb_qwen3.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
    embeddings = model.encode(
        singleturn["conversation"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


#### Nomic


In [ ]:
out_path = PROCESSED_DATA_DIR / "singleturn_emb_nomic.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)
    model.max_seq_length = 8192
    embeddings = model.encode(
        ("classification: " + singleturn["conversation"]).tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


### Review and check the embeddings

In [ ]:
# load embeddings
emb_gpt2 = np.load(PROCESSED_DATA_DIR / "singleturn_emb_gpt2.npy")
emb_qwen3 = np.load(PROCESSED_DATA_DIR / "singleturn_emb_qwen3.npy")
emb_nomic = np.load(PROCESSED_DATA_DIR / "singleturn_emb_nomic.npy")

# check
for name, emb in [("gpt2", emb_gpt2), ("qwen3", emb_qwen3), ("nomic", emb_nomic)]:
    assert emb.shape[0] == len(singleturn), f"{name} not row-aligned with singleturn!"
    assert np.isfinite(emb).all(), f"{name} has NaN/inf"
    print(f"{name:>5}: {emb.shape}")


### Generate Embeddings for Multi-turn

Waiting on Rachel's preprocessing

In [ ]:
# Load Data — same multiturn test files as 06_flo_baseline
mt = pd.read_csv(PROCESSED_DATA_DIR / "multiturn_X_test.csv")
mt["harm"] = pd.read_csv(PROCESSED_DATA_DIR / "multiturn_Y_test.csv").iloc[:, 0]

assert not mt["conversation"].str.strip().eq("").any()
print(f"{len(mt)} conversations, {mt['harm'].mean():.3f} harmful")

#### GPT-2

In [ ]:
out_path = PROCESSED_DATA_DIR / "multiturn_test_emb_gpt2.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModel.from_pretrained("gpt2").to(device)
    model.eval()

    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(mt), 8)):
            batch = mt["conversation"].iloc[i:i + 8].tolist()
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
            outputs = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1)
            pooled = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            all_embeddings.append(pooled.cpu().numpy())

    np.save(out_path, np.concatenate(all_embeddings, axis=0))


#### Qwen-3

In [ ]:
out_path = PROCESSED_DATA_DIR / "multiturn_test_emb_qwen3.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
    embeddings = model.encode(
        mt["conversation"].tolist(),
        batch_size=1,          # long conversations — avoid OOM
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


#### Nomic

In [ ]:
out_path = PROCESSED_DATA_DIR / "multiturn_test_emb_nomic.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)
    model.max_seq_length = 8192
    embeddings = model.encode(
        ("classification: " + mt["conversation"]).tolist(),
        batch_size=1,          # long conversations — avoid OOM
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


#### Review and Check

In [ ]:
for name in ("gpt2", "qwen3", "nomic"):
    emb = np.load(PROCESSED_DATA_DIR / f"multiturn_test_emb_{name}.npy")
    assert emb.shape[0] == len(mt), f"{name} not row-aligned with multiturn file!"
    assert np.isfinite(emb).all(), f"{name} has NaN/inf"
    print(f"{name:>5}: {emb.shape}")
